In [78]:
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, make_scorer
from catboost import CatBoostClassifier, Pool, EShapCalcType, EFeaturesSelectionAlgorithm
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')

In [64]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
ss = pd.read_csv('solution_format.csv')

train = train.drop("ID", axis=1)
df = train.copy()

In [65]:
grp_1 = train.filter(regex="_EventInd|index_").columns
grp_2 = train.filter(regex="_A|_F|_G").columns
grp_3 = list(set(train.columns).difference(grp_1).difference(grp_2))

In [73]:
df[grp_1]=df[grp_1].fillna(0)
df[grp_2]=df[grp_2].apply(str).fillna(df.mode().iloc[0])
df[grp_3]=df[grp_3].fillna(df.mean())

In [74]:
X = df.drop("Target", axis=1)
y = df.Target 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)

In [75]:
train_data=Pool(
    data=X_train,
    label=y_train,
    cat_features=grp_2.to_list()
)
eval_data=Pool(
    data=X_test,
    label=y_test,
    cat_features=grp_2.to_list()
)

In [112]:
cv=KFold(n_splits=5,shuffle=True,random_state=0)

In [113]:
# model=CatBoostRegressor(iterations=1000,learning_rate=0.05,depth=8)

# model = CatBoostClassifier(iterations=100, depth=8, learning_rate=0.1, loss_function='Logloss', custom_metric=['AUC'], random_seed=42)

model = CatBoostClassifier(iterations=100, learning_rate=0.1, depth=8, loss_function='Logloss', custom_metric=['AUC'], random_seed=1234)                

scores=[]
for i,j in enumerate(cv.split(X,y)):
    train=Pool(
        data=X.iloc[j[0]],
        label=y.iloc[j[0]],
        cat_features=grp_2.to_list()
    )
    eval=Pool(
        data=X.iloc[j[1]],
        label=y.iloc[j[1]],
        cat_features=grp_2.to_list()
    )
    model.fit(
        train,
        eval_set=eval,
        verbose=False
    )
    
    y_pred=model.predict(X.iloc[j[1]])
    score=f1_score(y.iloc[j[1]], y_pred, average='binary')
    scores.append(score)
    print(f"Fold {i}\t:{score}\n")

print(f"CV_SCORES_mean: {np.mean(scores)}")
print(f"CV_SCORES_std: {np.std(scores)}")

Fold 0	:0.7416413373860181

Fold 1	:0.7725321888412018

Fold 2	:0.7378048780487805

Fold 3	:0.7453237410071942

Fold 4	:0.7593123209169054

CV_SCORES_mean: 0.75132289324002
CV_SCORES_std: 0.012853981954864025


In [114]:
train=Pool(
    data=X_train,
    label=y_train,
    cat_features=grp_2.to_list()
)
eval=Pool(
    data=X_test,
    label=y_test,
    cat_features=grp_2.to_list()
)
model.fit(
    train,
    eval_set=eval,
    verbose=True
)

0:	learn: 0.6842904	test: 0.6877926	best: 0.6877926 (0)	total: 52.3ms	remaining: 5.18s
1:	learn: 0.6745329	test: 0.6834967	best: 0.6834967 (1)	total: 87.6ms	remaining: 4.29s
2:	learn: 0.6655021	test: 0.6788099	best: 0.6788099 (2)	total: 114ms	remaining: 3.69s
3:	learn: 0.6577628	test: 0.6722310	best: 0.6722310 (3)	total: 150ms	remaining: 3.61s
4:	learn: 0.6485125	test: 0.6695443	best: 0.6695443 (4)	total: 193ms	remaining: 3.67s
5:	learn: 0.6402652	test: 0.6652017	best: 0.6652017 (5)	total: 228ms	remaining: 3.57s
6:	learn: 0.6294345	test: 0.6606450	best: 0.6606450 (6)	total: 258ms	remaining: 3.43s
7:	learn: 0.6229090	test: 0.6586994	best: 0.6586994 (7)	total: 292ms	remaining: 3.35s
8:	learn: 0.6130891	test: 0.6535533	best: 0.6535533 (8)	total: 344ms	remaining: 3.48s
9:	learn: 0.6056645	test: 0.6532932	best: 0.6532932 (9)	total: 406ms	remaining: 3.66s
10:	learn: 0.5976043	test: 0.6474379	best: 0.6474379 (10)	total: 543ms	remaining: 4.39s
11:	learn: 0.5902988	test: 0.6448256	best: 0.64482

## Submission

In [97]:
test.drop("ID", axis=1, inplace=True)

In [98]:
test[grp_1]=test[grp_1].fillna(0)
test[grp_2]=test[grp_2].apply(str).fillna(df.mode().iloc[0])
test[[x for x in grp_3 if x!="Target"]]=test[[x for x in grp_3 if x!="Target"]].fillna(df.mean())

In [89]:
ss.head()

,ID,Target
0,2661,1
1,2662,1
2,2663,1
3,2664,1
4,2665,1


In [99]:
predict=model.predict(test)
ss.Target = predict
ss.head()

,ID,Target
0,2661,1
1,2662,1
2,2663,0
3,2664,1
4,2665,1


In [100]:
ss.to_csv('submission_1.csv', index = False)